[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jdsanch1/simrc/blob/master/02.%20Parte%202/11.%20Clase%2011/11Class%20NB.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jdsanch1/SimRC/master?labpath=02.%20Parte%202%2F11.%20Clase%2011%2F11Class%20NB.ipynb)

In [ ]:
import importlib, subprocess, sys

_required = ["yfinance", "pandas", "numpy", "matplotlib", "seaborn", "scipy", "sklearn", "statsmodels", "cvxpy"]
_missing  = [pkg for pkg in _required if importlib.util.find_spec(pkg) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _missing)

# Clase 11: Frontera eficiente de Markowitz con CVXPY

**[Juan Diego Sánchez Torres](https://www.researchgate.net/profile/Juan_Diego_Sanchez_Torres)**  
*Profesor*, [MAF ITESO](http://maf.iteso.mx/web/general/detalle?group_id=5858156)  
dsanchez@iteso.mx

## Objetivos de aprendizaje

- Implementar la frontera eficiente completa usando funciones reutilizables (`portfolio_func.py`).
- Usar **estimadores robustos** (Huber + Ledoit-Wolf) como insumos de la optimización.
- Comparar **simulación Monte Carlo** vs. **frontera eficiente** (CVXPY/DCP).
- Extender el portafolio con un **activo libre de riesgo** (bono).
- Entender las **condiciones KKT** y su interpretación financiera (Boyd & Vandenberghe, 2004, §5.5).

---

## Introducción teórica

### De las clases anteriores a esta

Esta clase integra los conceptos desarrollados en las Clases 4–10:

| Componente | Clase | Herramienta |
|-----------|:-----:|------------|
| Frontera eficiente (QP) | 4 | CVXPY `cp.quad_form` |
| Covarianza robusta (Σ) | 5 | `sklearn.covariance.ShrunkCovariance` |
| Media robusta (μ) | 6 | `statsmodels.robust.scale.Huber` |
| Monte Carlo | 9 | `np.random.dirichlet` |
| Bono (CML) | 10 | Extensión de Σ con ceros |

### Programación cuadrática paramétrica

La frontera eficiente se construye resolviendo una **familia paramétrica de QPs**. Para cada nivel de aversión al riesgo $\mu$:

$$
\min_{\mathbf{w}} \quad \mu \, \mathbf{w}^\top \Sigma \, \mathbf{w} - \hat{\boldsymbol{\mu}}^\top \mathbf{w} \qquad \text{s.a.} \quad \sum w_i = 1, \quad w_i \geq 0
$$

donde $\hat{\boldsymbol{\mu}}$ y $\hat{\Sigma}$ son los estimadores robustos. Al variar $\mu$ se traza toda la frontera eficiente.

### Funciones reutilizables (`portfolio_func.py`)

El módulo `portfolio_func.py` contiene las funciones del curso:
- `get_historical_closes()` — descarga datos con yfinance
- `calc_daily_returns()` — rendimientos logarítmicos
- `sim_mont_portfolio()` — Monte Carlo con Huber + Shrunk Covariance
- `optimal_portfolio()` — frontera eficiente con CVXPY (DCP)
- `optimal_portfolio_b()` — frontera con bono (activo libre de riesgo)

## 1. Descarga de datos y rendimientos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cvxpy as cp

sns.set_theme(style="darkgrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 100, "figure.figsize": (12, 5)})
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 10)

# Funciones del curso
import portfolio_func

In [ ]:
assets = ["AAPL", "MSFT", "AA", "AMZN", "KO", "QAI"]
closes = portfolio_func.get_historical_closes(assets, "2025-01-01", "2025-03-27")

fig, ax = plt.subplots()
(closes / closes.iloc[0] * 100).plot(ax=ax)
ax.set_title("Precios normalizados (base 100)")
ax.set_ylabel("Índice")
plt.tight_layout()

In [ ]:
daily_returns = portfolio_func.calc_daily_returns(closes)
daily_returns.describe()

---
## 2. Simulación Monte Carlo (100,000 portafolios)

Usamos `sim_mont_portfolio()` que internamente aplica:
- **Huber** para estimar $\hat{\boldsymbol{\mu}}$ (media robusta)
- **ShrunkCovariance** para estimar $\hat{\Sigma}$ (covarianza robusta)

In [ ]:
r = 0.0001  # Tasa libre de riesgo diaria
results_mc = portfolio_func.sim_mont_portfolio(daily_returns, 100000, r)
print(f"Portafolios simulados: {len(results_mc):,}")

In [ ]:
# Portafolios destacados
max_sharpe_mc = results_mc.iloc[results_mc["Sharpe"].idxmax()]
min_vol_mc = results_mc.iloc[results_mc["SD"].idxmin()]

print("Monte Carlo — Max Sharpe:")
print(f"  Rendimiento: {max_sharpe_mc['Returns']:.4f}, Riesgo: {max_sharpe_mc['SD']:.4f}, Sharpe: {max_sharpe_mc['Sharpe']:.4f}")
print(f"\nMonte Carlo — Min Volatilidad:")
print(f"  Rendimiento: {min_vol_mc['Returns']:.4f}, Riesgo: {min_vol_mc['SD']:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(results_mc.SD, results_mc.Returns, c=results_mc.Sharpe,
                     cmap="RdYlBu", s=3, alpha=0.3)
plt.colorbar(scatter, label="Ratio de Sharpe")
ax.scatter(max_sharpe_mc[1], max_sharpe_mc[0], marker="*", s=400,
           color="red", zorder=5, edgecolors="black", label="Max Sharpe (MC)")
ax.scatter(min_vol_mc[1], min_vol_mc[0], marker="*", s=400,
           color="green", zorder=5, edgecolors="black", label="Min Vol (MC)")
ax.set_xlabel("Riesgo (σ anualizada)")
ax.set_ylabel("Rendimiento esperado (anualizado)")
ax.set_title("Simulación Monte Carlo — 100,000 portafolios (Huber + ShrunkCov)")
ax.legend()
plt.tight_layout()

---
## 3. Frontera eficiente con CVXPY (DCP)

### Verificación DCP del problema

| Expresión CVXPY | Curvatura DCP | Justificación (Boyd §3.2) |
|---|---|---|
| `cp.quad_form(w, Σ)` | Convexa | $\Sigma \succeq 0$ (PSD) |
| `mu_vec @ w` | Afín | Producto con vector constante |
| `cp.sum(w) == 1` | Afín (igualdad) | Restricción lineal |
| `w >= 0` | Afín (desigualdad) | Restricción lineal |

`optimal_portfolio()` resuelve 5,000 QPs paramétricos con `cp.Parameter()` y `warm_start=True` para trazar la frontera completa eficientemente.

In [ ]:
N = 5000
results_optim = portfolio_func.optimal_portfolio(daily_returns, N, r)
print(f"Puntos en la frontera: {len(results_optim)}")

In [ ]:
# Portafolios óptimos
max_sharpe_optim = results_optim.iloc[results_optim["Sharpe"].idxmax()]
min_vol_optim = results_optim.iloc[results_optim["SD"].idxmin()]

print("CVXPY — Max Sharpe:")
print(f"  Rendimiento: {max_sharpe_optim['Returns']:.4f}, Riesgo: {max_sharpe_optim['SD']:.4f}, Sharpe: {max_sharpe_optim['Sharpe']:.4f}")
print(f"\nPesos óptimos:")
for asset in assets:
    print(f"  {asset}: {max_sharpe_optim[asset]:.4f}")

### Superposición: Monte Carlo + frontera eficiente

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# Nube MC
scatter = ax.scatter(results_mc.SD, results_mc.Returns, c=results_mc.Sharpe,
                     cmap="RdYlBu", s=3, alpha=0.2)
plt.colorbar(scatter, label="Sharpe (MC)")

# Frontera eficiente
ax.plot(results_optim.SD, results_optim.Returns, "b-", linewidth=2.5,
        label="Frontera eficiente (CVXPY)")

# Puntos destacados
ax.scatter(max_sharpe_optim[1], max_sharpe_optim[0], marker="*", s=400,
           color="red", zorder=5, edgecolors="black", label=f"Max Sharpe (S={max_sharpe_optim['Sharpe']:.3f})")
ax.scatter(min_vol_optim[1], min_vol_optim[0], marker="*", s=400,
           color="green", zorder=5, edgecolors="black", label=f"Min Vol (σ={min_vol_optim['SD']:.3f})")

ax.set_xlabel("Riesgo (σ anualizada)")
ax.set_ylabel("Rendimiento esperado (anualizado)")
ax.set_title("Monte Carlo vs. Frontera Eficiente de Markowitz (CVXPY/DCP)")
ax.legend(loc="upper left")
plt.tight_layout()

---
## 4. Inclusión de un bono (activo libre de riesgo)

`optimal_portfolio_b()` extiende la matriz de covarianza con una fila/columna de ceros para el bono y resuelve el mismo QP paramétrico. Esto produce la **Capital Market Line** (CML), derivada en la Clase 10.

In [ ]:
c0 = 0.000001  # Rendimiento diario del bono
results_optim_b = portfolio_func.optimal_portfolio_b(daily_returns, 10000, r, c0)
print(f"Puntos en la frontera con bono: {len(results_optim_b)}")

In [ ]:
max_sharpe_b = results_optim_b.iloc[results_optim_b["Sharpe"].idxmax()]
min_vol_b = results_optim_b.iloc[results_optim_b["SD"].idxmin()]

print("Con bono — Max Sharpe:")
print(f"  Rendimiento: {max_sharpe_b['Returns']:.6f}, Riesgo: {max_sharpe_b['SD']:.6f}")
print(f"  Sharpe: {max_sharpe_b['Sharpe']:.4f}")
print(f"\nPeso en BOND: {max_sharpe_b.get('BOND', 'N/A')}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

ax.plot(results_optim.SD, results_optim.Returns, "b-", linewidth=2,
        label="Sin bono")
ax.plot(results_optim_b.SD, results_optim_b.Returns, "r--", linewidth=2,
        label="Con bono (CML)")

ax.scatter(max_sharpe_optim[1], max_sharpe_optim[0], marker="*", s=300,
           color="blue", zorder=5, edgecolors="black")
ax.scatter(max_sharpe_b[1], max_sharpe_b[0], marker="D", s=150,
           color="red", zorder=5, edgecolors="black")

ax.set_xlabel("Riesgo (σ anualizada)")
ax.set_ylabel("Rendimiento esperado (anualizado)")
ax.set_title("Frontera eficiente: sin bono vs. con bono")
ax.legend()
plt.tight_layout()

---
## 5. Comparación de pesos óptimos

In [ ]:
comp = pd.DataFrame({
    "MC (max Sharpe)": [max_sharpe_mc[a] for a in assets],
    "CVXPY (max Sharpe)": [max_sharpe_optim[a] for a in assets],
}, index=assets)
comp["Diferencia"] = comp["CVXPY (max Sharpe)"] - comp["MC (max Sharpe)"]
comp

In [ ]:
comp[["MC (max Sharpe)", "CVXPY (max Sharpe)"]].plot(kind="bar", figsize=(10, 5))
plt.title("Pesos óptimos: Monte Carlo vs. CVXPY")
plt.ylabel("Peso")
plt.xticks(rotation=0)
plt.tight_layout()

---

## Navegación del curso

← **Anterior**: Clase 10: Activo libre de riesgo  
→ **Siguiente**: Clase 12: Optimización avanzada de portafolios

---

## 6. Referencias bibliográficas

- **Boyd, S. & Vandenberghe, L.** (2004). *Convex Optimization*. Cambridge University Press. — §3.2 (DCP), §4.4 (QP), §4.7.3 (paramétrica), §5.5 (KKT), §11.1 (punto interior).
- **Ledoit, O. & Wolf, M.** (2004). A well-conditioned estimator for large-dimensional covariance matrices. *Journal of Multivariate Analysis*, 88(2), 365–411.
- **Luenberger, D. G.** (2013). *Investment Science* (2nd ed.). Oxford University Press. — Cap. 6–8.
- **Markowitz, H.** (1952). Portfolio Selection. *The Journal of Finance*, 7(1), 77–91.
- **Sharpe, W. F.** (1964). Capital Asset Prices. *The Journal of Finance*, 19(3), 425–442.
- **Tsay, R. S.** (2010). *Analysis of Financial Time Series* (3rd ed.). Wiley.
- **Venegas Martínez, F.** (2008). *Riesgos financieros y económicos* (2a ed.). Cengage Learning.